In [5]:
import json
import pandas as pd
from collections import Counter
import os
import re
#,;/을 키워드에서 제거.
#모두 소문자 처리
#년도별로 파일 생성 및 중분류별로 키워드 빈도수 집계
#한 논문당 여러 키워드들이 존재하는데 이를 별도의 키워드들로 처리
#추가적으로, convolutional c~ network(CNN)과 같은 형식들은 괄호 안의 문자들을 제거하여 처리.
# =========================================================
# [설정] 입력 파일 및 결과 저장 폴더
# =========================================================
input_file_path = '../SSU_Datathon.json'
output_dir = '../result_by_year_keywords'

# =========================================================
# 1. 데이터 로드
# =========================================================
if not os.path.exists(input_file_path):
    print(f"오류: '{input_file_path}' 파일을 찾을 수 없습니다.")
    exit()

with open(input_file_path, 'r', encoding='utf-8') as f:
    try:
        raw_data = json.load(f)
    except json.JSONDecodeError:
        print("JSON 파일 형식이 올바르지 않습니다.")
        exit()

# NODE_LIST 구조 처리
if isinstance(raw_data, dict) and "NODE_LIST" in raw_data:
    data_list = raw_data["NODE_LIST"]
elif isinstance(raw_data, list):
    data_list = raw_data
else:
    data_list = [raw_data]

if not data_list:
    print("데이터가 비어있습니다.")
    exit()

df = pd.DataFrame(data_list)

# 필수 컬럼 확인
required_columns = ['PBSH', 'NODE_CLSS_02', 'KYWD']
for col in required_columns:
    if col not in df.columns:
        print(f"오류: 필수 컬럼 '{col}' 이(가) 누락되었습니다.")
        exit()

# =========================================================
# 2. 전처리
# =========================================================
df['Year'] = df['PBSH'].fillna('').astype(str).str[:4]
df['NODE_CLSS_02'] = df['NODE_CLSS_02'].fillna('미분류')
df['KYWD'] = df['KYWD'].fillna('')

# 유효한 연도만 사용
years = sorted([y for y in df['Year'].unique() if y.isdigit() and len(y) == 4])

# 결과 폴더 생성
os.makedirs(output_dir, exist_ok=True)
print(f"'{output_dir}' 폴더에 연도별 키워드 분석 파일을 생성합니다...")

# =========================================================
# 3. 연도별 파일 생성 → 파일 내부 중분류 분리
# =========================================================
for year in years:
    year_df = df[df['Year'] == year]
    filename = os.path.join(output_dir, f"{year}_keywords.txt")

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"==========================================================\n")
        f.write(f"      [ {year}년도 키워드 분석 보고서 ]\n")
        f.write(f"      총 논문 수: {len(year_df)}건\n")
        f.write(f"==========================================================\n\n")

        # 중분류별 그룹화 (논문 수 기준 정렬)
        class_counts = year_df['NODE_CLSS_02'].value_counts()

        for class_name, paper_count in class_counts.items():
            class_df = year_df[year_df['NODE_CLSS_02'] == class_name]

            all_keywords = []
            for kw_str in class_df['KYWD']:
                text = str(kw_str)

                # 1. 괄호 안 문자 제거
                text = re.sub(r'\([^)]*\)', '', text)

                # 2. , ; / 기준 분리
                tokens = re.split(r'[,;/]', text)

                # 3. 소문자 + 공백 정리
                tokens = [t.strip().lower() for t in tokens if t.strip()]
                all_keywords.extend(tokens)

            keyword_counter = Counter(all_keywords)

            f.write(f"\n{'#'*80}\n")
            f.write(f"[중분류] {class_name} (논문 {paper_count}건)\n")
            f.write(f"{'#'*80}\n")

            if not keyword_counter:
                f.write("  (키워드 없음)\n")
                continue

            for rank, (kw, freq) in enumerate(keyword_counter.most_common(), 1):
                f.write(f"  {rank}. {kw} ({freq})\n")

    print(f" -> {filename} 저장 완료 ({len(year_df)}건)")

print("\n모든 연도별 키워드 분석이 완료되었습니다.")


'../result_by_year_keywords' 폴더에 연도별 키워드 분석 파일을 생성합니다...
 -> ../result_by_year_keywords\2021_keywords.txt 저장 완료 (12774건)
 -> ../result_by_year_keywords\2022_keywords.txt 저장 완료 (12659건)
 -> ../result_by_year_keywords\2023_keywords.txt 저장 완료 (12502건)
 -> ../result_by_year_keywords\2024_keywords.txt 저장 완료 (12965건)
 -> ../result_by_year_keywords\2025_keywords.txt 저장 완료 (11299건)

모든 연도별 키워드 분석이 완료되었습니다.
